## Overview
To extend LLM's knowledge outside of the data it has been trained on, we use RAG (Retrieval Augmented Generation). Using RAG, we can fnid relevant information from our data and inject it into the LLM prompt. To search for relevant information, we can make use of:
- full text search
- vector search
- combination of both

RAG essentially has two steps:
- **Ingestion:** is offline pipeline that takes raw source content (docs, PDFs, wikis, etc.) and turns it into something an LLM can efficiently search over at query time. It involves the following steps:
    1. Load: read raw content from its source (text files, PDFs, web pages, databases) into a normalized in-memory representation (represented as `Document` in LangChain4j).
    2. Split (chunk): break each document into smaller pieces (`TextSegments`). This matters because the challenge lies in ensuring each segment provides sufficient context/information for the LLM to understand it, and missing context can lead to the LLM misinterpreting the given segment and hallucinating, therefore splitters commonly use overlapping windows to preserve context across chunk boundaries.
    3. Transform (optional): clean, enrich, or reformat each segment before embedding. As an example, stripping boilerplate, adding metadata like source/section, or summarizing.
    4. Embed: convert each text segment into a vector (an `Embedding`) using an embedding model, so semantic similarity becomes a distance calculation.
    5. Store: persist those vectors (plus their originating text and metadata) into a vector database (`EmbeddingStore`) so they can be searched later by similarity.

- **Retrieval:** turns user question to search query and finds most relevant previously ingested data using that. Then it injects this information into the LLM prompt. It involves the following steps:
    1. Query transformation: `QueryTransformer` transforms query into one or multiple querys. This step exists because the raw user message isn't always the best search query. Common use case is query compression, where a `ChatModel` is used to rephrase the query concisely, preserving its intent while improving retrieval accuracy. It can also expand one question into several sub-queries for broader coverage.
    2. Query routing: each `Query` is routed by the `QueryRouter` to one or more `ContentRetrievers`. This matters when you have multiple sources (a vector store, a SQL database, a web search engine, etc.).
    3. Content retrieval: each `ContentRetriever` retrieves relevant `Contents` for each `Query`. The most common implementation, `EmbeddingStoreContentRetriever`, embeds the query with the same `EmbeddingModel` used at ingestion time and does a similarity search against the `EmbeddingStore`.
    4. Aggregation/Re-ranking: the `ContentAggregator` combines all retrieved `Contents` into a single final ranked list.
    5. Content injection: this list of `Contents` is injected into the original `UserMessage`, via a `ContentInjector`, so the LLM sees the retrieved passages alongside the original question.

## Terms
**Document:** represents textual contents of a document like text file or PDF file. It is loaded using a document loader.

In [ ]:
Document document = Document document = FileSystemDocumentLoader
                .loadDocument(Path.of("C:\\temp\\example.txt"));

/* Other example loaders
- ClassPathDocumentLoader
- UrlDocumentLoader
- AmazonS3DocumentLoader
*/

// Or if you have plain string
Document strDocument = Document.from("Here is some text.");

In the above example a document parser (in this case `TextDocumentParser`) was used to read the contents. For documents like PDF, docx, etc, we need to use specialized document parsers:

In [ ]:
Document pdfDocument = FileSystemDocumentLoader
                .loadDocument(Path.of("C:\\temp\\example.pdf"),
                        new ApacheTikaDocumentParser()); // will convert binary pdf into plain text

/* Other DocumentParser
- MarkdownDocumentParser 
- YamlDocumentParser
- ApachePoiDocumentParser
*/

`Document` contains two parts - text and metadata. Metadata is effectively a key-value pair.

In [ ]:
document.text();
document.metadata();

// add information to metadata
document.metadata().put("author", "John Doe");

**Splitting:** as we read earlier `Document` needs to be split into smaller size pieces because:
- if we keep the whole document intact, then retrieval would also return the whole document. Even though only one paragraph was relevant.
- retrieving and injecting smaller chunks also helps in keeping cost in control

Chunked document would look like:
```
Chunk 1: ## 1. Visual Theme & Atmosphere\n\nMastercard's experience reads like a warm, editorial magazine built from soft stone and signal orange...
Chunk 2: orange. The canvas is a muted putty-cream (`#F3F0EE`) — not white, not gray, but a color that feels like the paper of a premium annual report...
```

We make use of `DocumentSplitter` for this job:

In [ ]:
// Recursive means it tries to split at the biggest, most meaningful boundary first (paragraphs, sentences), 
// and only drops down to smaller boundaries (words) when a chunk is still too large.
DocumentSplitter splitter = DocumentSplitters.recursive(500, 50);

// 500 - max size of each chunk. Smaller is more precise, but more fragmented.
//       larger means more context in each chunk, but noisier retrieval
// 50 - overlap. In this case it means it chunk repeats 50 characters of previous

Splitting document results in `TextSegment`s which is text content of one chunk + metadata inherited from the parent document.

In [ ]:
List<TextSegment> chunks = splitter.split(document);

**Embedding and Embedding Model:** an embedding is a vector of numbers that represents meaning of text content. We use an embedding model to convert text to embedding. Text with similar meaning are closer together in the vector space.

In [ ]:
if (!chunks.isEmpty()) {
    EmbeddingModel model = new AllMiniLmL6V2EmbeddingModel();
    List<Embedding> embeddings = new ArrayList<>();
    for (TextSegment segment : chunks) {
        Embedding embedding = model.embed(segment).content();
        // embedding.vector()  -> underlying vector
        embeddings.add(embedding);
    }
}

// Use EmbeddingRequest.builder() for more control over how embedding is generated

Some models support both text and image content combined to single embedding:

In [ ]:
EmbeddingResponse response = embeddingModel.embed(EmbeddingRequest.builder()
    .input(TextContent.from("a photo of a cat"), ImageContent.from("https://example.com/cat.png"))
    .build());

Some commonly used embedding models:

| Model | Provider / Hosting | Cost | Typical Dimensions | Notes |
|-------|---------------------|------|--------------------|-------|
| AllMiniLmL6V2 (in-process) | Local (ONNX, in-JVM) | Free | 384 | Fast, tiny, zero network latency; good baseline for prototyping or offline/air-gapped apps; lower semantic quality than larger models. |
| BGE-large-en / multilingual (in-process) | Local (ONNX) | Free | 1024 | Stronger retrieval quality than MiniLM, still local—but with a heavier memory/CPU footprint and slower per-call performance. |
| OpenAI `text-embedding-3-small` | API | Paid (low cost) | 1536 (configurable, can shrink) | Strong general-purpose quality, simple to set up, but adds network latency and per-token cost; supports dimension reduction to save storage. |
| OpenAI `text-embedding-3-large` | API | Paid (higher) | 3072 (configurable) | Best OpenAI embedding quality; more expensive and slightly slower than `-small`; overkill for many RAG use cases. |
| Azure OpenAI (same models) | API (Azure-hosted) | Paid (enterprise billing) | Same as OpenAI | Same models as OpenAI, but routed through Azure; useful if you already have Azure infrastructure or compliance requirements. |
| Amazon Bedrock Titan / Cohere | API (AWS-hosted) | Paid (AWS billing) | Varies by model | Good fit if you're already on AWS (IAM auth, VPC, billing consolidation); Cohere-on-Bedrock variants offer strong multilingual support. |
| Cohere (direct) | API | Paid | 384–1024 (depending on model) | Known for strong multilingual embeddings and dedicated rerank models; pairs well with `ReRankingContentAggregator` in retrieval. |
| Voyage AI | API | Paid | Varies | Designed specifically for retrieval/RAG use cases; often benchmarks well on domain-specific retrieval tasks. |
| Ollama-hosted models | Local (separate Ollama server process, not in-JVM) | Free (self-hosted compute) | Varies by model | Good middle ground: local/private like in-process, but supports many open models through Ollama without requiring ONNX conversion. |

**Embedding Store:** is where we store embeddings. This store also allows us to search similar embedding. We can chose to only store `Embedding` or store corresponding `TextSegment` as well.

In [ ]:
EmbeddingStore<TextSegment> store = new InMemoryEmbeddingStore<>();
store.addAll(embeddings, chunks);

**Query:** represents user search term

In [ ]:
Query query = Query.from("Retrieval Augmentation");

// Or query = Query.from(text, metadata); if we have a relevant metadata

This query can be transformed using `QueryTransformer` to generate one or more `Query` objects. As an example, `CompressingQueryTransformer` uses an LLM to compress the given `Query` and previous conversation into a standalone `Query`:

In [ ]:
QueryTransformer queryTransformer = new CompressingQueryTransformer(chatModel);
Collection<Query> queries = queryTransformer.transform(query);

/* Here is how the interface looks like:
public interface QueryTransformer {
    Collection<Query> transform(Query query);
}
*/

**Content Retriever:** against the `Query`, we use a `ContentRetriever` to return list of `Content` which is a wrapper around retrieved data. The content is retrieved from underlying data source as we saw earlier, can be a embedding store.

In [ ]:
ContentRetriever contentRetriever = EmbeddingStoreContentRetriever.builder()
    .embeddingStore(store)
    .embeddingModel(model)
    .maxResults(5)
    .minScore(0.3)
    .build();

For each query, we can specify a particular retriever to use. This can be done using a `QueryRouter`:

In [ ]:
QueryRouter queryRouter = new DefaultQueryRouter(List.of(contentRetriever));

/* This interface looks like
public interface QueryRouter {
    Collection<ContentRetriever> route(Query query);
}
*/

Since a `Query` can be routed to multiple retrievers and one retriever can return multiple `Content`, we need a way to aggregate these results. This can be done using a `ContentAggregator`. Its method signature looks like:

In [ ]:
List<Content> aggregate(Map<Query, Collection<List<Content>>> queryToContents);

/*
The map looks like:
Query1 → [ [Content, Content, ...] from Retriever A,
           [Content, Content, ...] from Retriever B ]

Query2 → [ [Content, Content, ...] from Retriever A,
           [Content, Content, ...] from Retriever B ]
*/

`DefaultContentAggregator` merges the above result in two stages via Reciprocal Rank Fusion:
- Stage 1: for each `Query`, all `List<Content>` retrieved with that query (i.e., from its different routed retrievers) are merged into one `List<Content>`.
- Stage 2: all of those per-query merged lists (one per `Query`) are merged into the single final `List<Content>`.

Other aggregator like `ReRankingContentAggregator` uses a `ScoringModel`, like *Cohere*, to perform re-ranking.

**Content Injector:** is responsible for injecting of `Content`s returned by `ContentAggregator` into the `UserMessage`. An implementation `DefaultContentInjector` is the default implementation of `ContentInjector` that simply appends `Content`s to the end of a `UserMessage` with the prefix *Answer using the following information:*. We can customize it:

In [ ]:
DefaultContentInjector.builder()
    // Must contain {{userMessage}} and {{contents}}
    .promptTemplate(PromptTemplate.from("{{userMessage}}\n{{contents}}"))
    .build();

/*
Add below to builder to add Content's source metadata key in the prompt
.metadataKeysToInclude(List.of("source"))
*/

**Retrieval Augmentor:** orchestrator interface for the entire retrieval pipeline. It is the single entry point `AiServices` calls to go from "a user's message" to "a user's message enriched with relevant content". It looks like:

In [ ]:
public interface RetrievalAugmentor {
    UserMessage augment(UserMessage userMessage, Metadata metadata);
}

Sample initialization:

In [ ]:
RetrievalAugmentor retrievalAugmentor = DefaultRetrievalAugmentor.builder()
    .queryTransformer(queryTransformer)
    .queryRouter(queryRouter)
    .contentAggregator(contentAggregator)
    .contentInjector(contentInjector)
    .build();

Every time an AI Service method is invoked, the configured `RetrievalAugmentor` is called to augment the current `UserMessage` before it is sent to the LLM. The augmentor orchestrates the full RAG pipeline:
1. Convert the `UserMessage` into a `Query`
2. Transform the query (e.g., compress conversation history, expand synonyms)
3. Route the query to one or more `ContentRetrievers`
4. Retrieve relevant `Content` from data sources (embedding stores, web search, SQL DB, etc.)
5. Aggregate and rank the retrieved content
6. Inject the content into the original `UserMessage` using a `ContentInjector`
7. Send the augmented message to the LLM for generation

we can configure this via two builder methods on `AiServices`:
- `contentRetriever(ContentRetriever)`: a shortcut that wraps a single retriever in a `DefaultRetrievalAugmentor`
- `retrievalAugmentor(RetrievalAugmentor)`: the full-power option for custom pipelines 